# LeetCode 1621 — Number of Sets of K Non-Overlapping Line Segments bằng Python

Link đề: https://leetcode.com/problems/number-of-sets-of-k-non-overlapping-line-segments/ · Độ khó: **Medium** · Chủ đề: Math, Dynamic Programming, Combinatorics, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def numberOfSets(self, n: int, k: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Mô hình hóa bài đếm hình học (đoạn thẳng trên trục số) thành bài đếm **cấu hình rời rạc**: chọn đầu mút, hoặc gán trạng thái cho từng **khe** giữa hai điểm liên tiếp.
2. Viết được **vét cạn** sinh mọi tập đoạn để có "đáp án chuẩn" kiểm tra các cách nhanh.
3. Xây **quy hoạch động** từ `O(n²·k)` (thử mọi điểm kết thúc) → `O(n·k)` (tổng hậu tố / tách trạng thái "đang mở – đã đóng").
4. Hiểu phép **song ánh** biến bài toán thành tổ hợp `C(n + k − 1, 2k)` và tính tổ hợp **modulo số nguyên tố** bằng nghịch đảo Fermat.
5. Tránh các bẫy: quên cho phép chung đầu mút, chia trong modulo, đệ quy quá sâu, lệch chỉ số `n` điểm ↔ `n − 1` khe.

## 2. Đề bài & ví dụ

Có `n` điểm trên trục số, điểm thứ `i` (từ `0` tới `n − 1`) nằm ở tọa độ `x = i`. Hãy đếm số cách vẽ **đúng `k`** đoạn thẳng **không chồng lên nhau** sao cho:
- mỗi đoạn phủ **ít nhất hai điểm** (tức đoạn `[l, r]` với `l < r`);
- hai đầu mút của mỗi đoạn là **số nguyên** (trùng với các điểm);
- `k` đoạn **không cần** phủ hết `n` điểm;
- các đoạn **được phép chung đầu mút** (ví dụ `(0,1)` và `(1,3)` cùng tồn tại).

Vì đáp án rất lớn nên trả về theo **modulo `10⁹ + 7`**.

**Ví dụ 1:** `n = 4, k = 2` → `5`. Năm cách: `{(0,2),(2,3)}`, `{(0,1),(1,3)}`, `{(0,1),(2,3)}`, `{(1,2),(2,3)}`, `{(0,1),(1,2)}`.

**Ví dụ 2:** `n = 3, k = 1` → `3`. Ba cách: `{(0,1)}`, `{(0,2)}`, `{(1,2)}`.

**Ví dụ 3:** `n = 30, k = 7` → `796297179`. Tổng số cách thật là `3796297200`, lấy modulo `10⁹ + 7` được `796297179`.

**Ràng buộc:**
- `2 ≤ n ≤ 1000`
- `1 ≤ k ≤ n − 1`

"Không chồng lên nhau" nghĩa là hai đoạn không có phần chung dài hơn một điểm: `(0,2)` và `(1,3)` chồng nhau (chung khúc `[1,2]`), còn `(0,2)` và `(2,3)` chỉ chạm nhau ở điểm 2 nên hợp lệ.

In [1]:
import random
import time
import sys
from math import comb
from functools import lru_cache
from itertools import combinations

MOD = 10**9 + 7

examples = [
    ((4, 2), 5),
    ((3, 1), 3),
    ((30, 7), 796297179),
]
for (n, k), want in examples:
    print(f"n={n:>2}, k={k} -> kỳ vọng {want}")

n= 4, k=2 -> kỳ vọng 5
n= 3, k=1 -> kỳ vọng 3
n=30, k=7 -> kỳ vọng 796297179


## 3. Kiến thức nền

### 3.1 Biểu diễn một đoạn và kiểm tra "không chồng nhau"

Một đoạn là cặp `(l, r)` với `0 ≤ l < r ≤ n − 1`. Nếu sắp các đoạn theo đầu trái, điều kiện không chồng nhau trở thành: **đầu phải đoạn trước ≤ đầu trái đoạn sau** (`r₁ ≤ l₂`). Dấu `≤` (chứ không phải `<`) chính là "được chung đầu mút".

In [2]:
def non_overlapping(segs):
    """segs: danh sách (l, r) bất kỳ; trả True nếu đôi một không chồng nhau."""
    s = sorted(segs)
    return all(s[i][1] <= s[i + 1][0] for i in range(len(s) - 1))

print(non_overlapping([(0, 2), (2, 3)]))   # chạm nhau ở 2 -> hợp lệ
print(non_overlapping([(0, 2), (1, 3)]))   # chung khúc [1,2] -> chồng
print(non_overlapping([(1, 2), (0, 1)]))   # thứ tự nhập không quan trọng

True
False
True


### 3.2 `itertools.combinations` và `math.comb`

- `combinations(iterable, r)` sinh mọi bộ `r` phần tử **theo thứ tự tăng dần của chỉ số**, không lặp — rất hợp để vét cạn "chọn một tập".
- `math.comb(n, r)` = `C(n, r)` = số cách chọn `r` phần tử từ `n` phần tử (số nguyên lớn tùy ý của Python, không tràn).

In [3]:
pts = range(4)
all_segs = list(combinations(pts, 2))        # mọi đoạn (l, r) với l < r trên 4 điểm
print("số đoạn đơn trên 4 điểm:", len(all_segs), all_segs)
print("C(4,2) =", comb(4, 2), "| C(36,14) =", comb(36, 14))

số đoạn đơn trên 4 điểm: 6 [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
C(4,2) = 6 | C(36,14) = 3796297200


### 3.3 Số học modulo và nghịch đảo Fermat

Với số nguyên tố `p = 10⁹ + 7`:
- `(a + b) % p`, `(a · b) % p` giữ được tính đúng khi lấy modulo từng bước.
- **Phép chia không làm trực tiếp được**: `(a / b) mod p` phải tính bằng `a · b^(p−2) mod p` (định lý Fermat nhỏ: `b^(p−1) ≡ 1`). Trong Python: `pow(b, p - 2, p)`.

In [4]:
p = MOD
b = 123456
inv_b = pow(b, p - 2, p)
print("b * inv_b % p =", b * inv_b % p)          # phải bằng 1
# Ví dụ: 10! / (3! * 7!) = C(10,3) = 120, tính dưới modulo
f10, f3, f7 = 3628800 % p, 6, 5040
print(f10 * pow(f3, p - 2, p) % p * pow(f7, p - 2, p) % p)

b * inv_b % p = 1
120


### 3.4 Ghi nhớ kết quả: `functools.lru_cache`

Đệ quy có nhớ (memoization) tránh tính lại cùng một trạng thái. `@lru_cache(maxsize=None)` lưu mọi kết quả theo tham số. Lưu ý giới hạn độ sâu đệ quy mặc định của Python khoảng `1000` — đây sẽ là một bẫy ở mục 13.

In [5]:
@lru_cache(maxsize=None)
def fib(i):
    return i if i < 2 else fib(i - 1) + fib(i - 2)

print(fib(80), "| recursion limit hiện tại:", sys.getrecursionlimit())

23416728348467685 | recursion limit hiện tại: 1000


### 3.5 Mảng 2 chiều đúng cách

`[[0] * (k + 1) for _ in range(n + 1)]` tạo `n + 1` hàng **độc lập**. Không dùng `[[0] * (k + 1)] * (n + 1)` vì mọi hàng trỏ cùng một list.

In [6]:
good = [[0] * 3 for _ in range(2)]
bad = [[0] * 3] * 2
good[0][0] = 1; bad[0][0] = 1
print("good:", good, "| bad:", bad)

good: [[1, 0, 0], [0, 0, 0]] | bad: [[1, 0, 0], [1, 0, 0]]


## 4. Ý tưởng nền tảng / trực giác

Tính tay với `n = 4` (điểm 0, 1, 2, 3), `k = 2`.

Có 6 đoạn đơn: `(0,1) (0,2) (0,3) (1,2) (1,3) (2,3)`. Ghép hai đoạn, sắp theo đầu trái, cần `r₁ ≤ l₂`:

| đoạn 1 | đoạn 2 có thể (`l₂ ≥ r₁`) | số cách |
|---|---|---|
| (0,1) | (1,2), (1,3), (2,3) | 3 |
| (0,2) | (2,3) | 1 |
| (0,3) | — | 0 |
| (1,2) | (2,3) | 1 |
| (1,3), (2,3) | — | 0 |

Tổng `3 + 1 + 1 = 5` ✔.

**Góc nhìn "khe":** giữa 4 điểm có 3 khe `g₁=[0,1]`, `g₂=[1,2]`, `g₃=[2,3]`. Mỗi đoạn là một **dãy khe liên tiếp** (≥ 1 khe). Hai đoạn khác nhau không dùng chung khe, nhưng có thể nằm sát nhau. Vì vậy mỗi khe ở một trong ba trạng thái:
- `.` trống,
- `S` = **bắt đầu** một đoạn mới,
- `C` = **nối tiếp** đoạn đang có ở khe ngay trước (chỉ được khi khe trước không trống).

Số đoạn = số chữ `S`. Với `n = 4, k = 2`: các chuỗi 3 ký tự có đúng 2 chữ `S` và không có `C` đứng sau `.` hoặc đứng đầu: `SS.`, `S.S`, `.SS`, `SCS`, `SSC` → 5 ✔. Góc nhìn này sẽ dẫn thẳng tới DP `O(n·k)`.

In [7]:
from itertools import product

def gap_strings(n, k):
    res = []
    for s in product(".SC", repeat=n - 1):
        if s.count("S") != k:
            continue
        ok = all(not (c == "C" and (i == 0 or s[i - 1] == ".")) for i, c in enumerate(s))
        if ok:
            res.append("".join(s))
    return res

def gaps_to_segments(s):
    segs, cur = [], None
    for i, c in enumerate(s):          # khe i nối điểm i và i+1
        if c == "S":
            if cur: segs.append(tuple(cur))
            cur = [i, i + 1]
        elif c == "C":
            cur[1] = i + 1
        else:
            if cur: segs.append(tuple(cur)); cur = None
    if cur: segs.append(tuple(cur))
    return segs

for s in gap_strings(4, 2):
    print(s, "->", gaps_to_segments(s))

.SS -> [(1, 2), (2, 3)]
S.S -> [(0, 1), (2, 3)]
SS. -> [(0, 1), (1, 2)]
SSC -> [(0, 1), (1, 3)]
SCS -> [(0, 2), (2, 3)]


## 5. Cách 1 — Brute Force / baseline: sinh mọi tập `k` đoạn

**Ý tưởng:** chọn các đoạn **theo thứ tự từ trái sang phải**. Đoạn đầu tiên có đầu trái `l ≥ 0`; mỗi đoạn sau phải có đầu trái `≥` đầu phải đoạn trước. Đệ quy `go(start, left)`: còn `left` đoạn cần đặt, đầu trái tiếp theo phải `≥ start`.

Vì luôn chọn theo thứ tự tăng của đầu trái nên mỗi tập được sinh **đúng một lần**.

In [8]:
def brute_sets(n, k):
    """Trả về danh sách mọi tập k đoạn (để xem tận mắt)."""
    out = []
    def go(start, left, chosen):
        if left == 0:
            out.append(list(chosen)); return
        for l in range(start, n):
            for r in range(l + 1, n):
                chosen.append((l, r))
                go(r, left - 1, chosen)       # đoạn sau bắt đầu từ r trở đi (cho chung đầu mút)
                chosen.pop()
    go(0, k, [])
    return out

def brute_count(n, k):
    return len(brute_sets(n, k)) % MOD

print(brute_sets(4, 2))
print(brute_sets(3, 1))
for (n, k), want in examples[:2]:
    print((n, k), brute_count(n, k), "kỳ vọng", want)

[[(0, 1), (1, 2)], [(0, 1), (1, 3)], [(0, 1), (2, 3)], [(0, 2), (2, 3)], [(1, 2), (2, 3)]]
[[(0, 1)], [(0, 2)], [(1, 2)]]
(4, 2) 5 kỳ vọng 5
(3, 1) 3 kỳ vọng 3


In [9]:
# Kiểm chứng chéo: brute force bằng combinations + non_overlapping (chậm hơn nữa nhưng "ngây thơ" nhất)
def brute_naive(n, k):
    segs = list(combinations(range(n), 2))
    return sum(1 for pick in combinations(segs, k) if non_overlapping(pick))

for n in range(2, 7):
    for k in range(1, n):
        assert brute_naive(n, k) == brute_count(n, k)
print("hai kiểu vét cạn khớp nhau với n ≤ 6")

hai kiểu vét cạn khớp nhau với n ≤ 6


**Độ phức tạp:** số lời gọi tỉ lệ với số tập hợp lệ — tức chính đáp án, có thể cỡ `C(n + k − 1, 2k)` (với `n = 30, k = 7` đã là `3,8·10⁹`). Hoàn toàn không dùng được với `n = 1000`.

**Nút thắt:** ta **liệt kê** từng cấu hình trong khi chỉ cần **đếm**. Nhiều nhánh đệ quy có cùng "phần còn lại" (cùng `start`, cùng `left`) nhưng bị tính lại từ đầu ⇒ gợi ý ghi nhớ theo trạng thái `(start, left)`.

In [10]:
for n, k in [(8, 3), (10, 4), (12, 5)]:
    t = time.perf_counter()
    c = brute_count(n, k)
    print(f"n={n}, k={k}: {c:>6} tập, {time.perf_counter() - t:.3f}s")

n=8, k=3:    210 tập, 0.000s
n=10, k=4:   1287 tập, 0.001s
n=12, k=5:   8008 tập, 0.005s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — trạng thái đệ quy chỉ phụ thuộc `(start, left)`:** số cách đặt `left` đoạn mà mọi đầu trái `≥ start` không phụ thuộc các đoạn đã chọn trước đó. Đặt `F(i, j)` = số cách đặt `j` đoạn trên các điểm `i..n−1`:
- Nếu **không** có đoạn nào bắt đầu tại `i`: `F(i + 1, j)`.
- Nếu có đoạn bắt đầu tại `i`, kết thúc tại `e` (`i < e ≤ n − 1`): `F(e, j − 1)`.
- `F(i, 0) = 1` (không cần đặt gì nữa — một cách), `F(n − 1, j ≥ 1) = 0` (một điểm không đủ tạo đoạn).

⇒ `F(i, j) = F(i+1, j) + Σ_{e=i+1}^{n−1} F(e, j−1)`. Ghi nhớ cho `O(n²·k)`.

**Insight 2 — tổng liên tiếp ⇒ tổng hậu tố:** phần `Σ_{e>i} F(e, j−1)` là tổng hậu tố của cột `j − 1` ⇒ tính trong `O(1)` ⇒ `O(n·k)`.

**Insight 3 — góc nhìn khe (mục 4):** quét `n − 1` khe từ trái sang phải với hai trạng thái "khe vừa rồi **trống**" (`a[j]`) và "khe vừa rồi **được phủ**" (`b[j]`), `j` = số đoạn đã mở:
- Nếu khe mới **trống**: từ cả `a[j]` và `b[j]` → `a[j]`.
- Nếu khe mới **bắt đầu đoạn**: từ `a[j−1]`, `b[j−1]` → `b[j]`.
- Nếu khe mới **nối tiếp**: chỉ từ `b[j]` → `b[j]`.

**Insight 4 — song ánh tổ hợp:** một tập `k` đoạn ↔ dãy đầu mút `x₁ < x₂ ≤ x₃ < x₄ ≤ … ≤ x₂ₖ₋₁ < x₂ₖ`. Cộng thêm `t − 1` vào cả hai đầu mút của đoạn thứ `t` thì mọi dấu `≤` thành `<`, giá trị chạy trong `0..n + k − 2` (`n + k − 1` giá trị) ⇒ đáp án là **`C(n + k − 1, 2k)`**.

Quy tắc "nếu… thì…":
- **Nếu** bài đếm "chọn các đoạn rời nhau trên đường thẳng" **thì** quét trái → phải với trạng thái "đang trong đoạn / không".
- **Nếu** công thức DP có `Σ` trên một dải chỉ số liên tiếp **thì** thay bằng prefix/suffix sum.
- **Nếu** ràng buộc là dãy tăng có lẫn `<` và `≤` **thì** thử "dịch" để thành dãy tăng nghiêm ngặt ⇒ tổ hợp.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — DP top-down `F(i, j)` thử mọi điểm kết thúc, `O(n²·k)`

Từng đoạn code:
1. `F(i, 0) = 1`: đã đặt đủ đoạn.
2. `i ≥ n − 1` mà còn `j ≥ 1` đoạn: không còn chỗ ⇒ 0.
3. Cộng nhánh "bỏ qua điểm `i`" và mọi nhánh "đoạn `[i, e]`".

In [11]:
def count_memo(n, k):
    @lru_cache(maxsize=None)
    def F(i, j):
        if j == 0:
            return 1
        if i >= n - 1:
            return 0
        total = F(i + 1, j)                    # không có đoạn bắt đầu tại i
        for e in range(i + 1, n):              # đoạn [i, e]
            total += F(e, j - 1)
        return total % MOD
    return F(0, k)

for (n, k), want in examples:
    print((n, k), count_memo(n, k), "kỳ vọng", want)

(4, 2) 5 kỳ vọng 5
(3, 1) 3 kỳ vọng 3
(30, 7) 796297179 kỳ vọng 796297179


Bản lặp (bottom-up) tránh đệ quy sâu: duyệt `i` giảm dần từ `n − 1` về `0`, mỗi `i` duyệt mọi `j`.

In [12]:
def count_n2k(n, k):
    F = [[0] * (k + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        F[i][0] = 1
    for i in range(n - 2, -1, -1):
        for j in range(1, k + 1):
            total = F[i + 1][j]
            for e in range(i + 1, n):
                total += F[e][j - 1]
            F[i][j] = total % MOD
    return F[0][k]

for (n, k), want in examples:
    print((n, k), count_n2k(n, k), "kỳ vọng", want)

(4, 2) 5 kỳ vọng 5
(3, 1) 3 kỳ vọng 3
(30, 7) 796297179 kỳ vọng 796297179


Code nộp LeetCode của Cách 2 (đúng nhưng **TLE** khi `n = 1000`, `k` lớn — khoảng `n²k/2 ≈ 5·10⁸` phép cộng):

```python
class Solution:
    def numberOfSets(self, n: int, k: int) -> int:
        MOD = 10**9 + 7
        F = [[0] * (k + 1) for _ in range(n + 1)]
        for i in range(n + 1):
            F[i][0] = 1
        for i in range(n - 2, -1, -1):
            for j in range(1, k + 1):
                total = F[i + 1][j]
                for e in range(i + 1, n):
                    total += F[e][j - 1]
                F[i][j] = total % MOD
        return F[0][k]
```

**Độ phức tạp:** Time `O(n²·k)`, Space `O(n·k)`.

### 7.2 Cách 3 — DP + tổng hậu tố, `O(n·k)`

Đặt `Suf[i][j] = Σ_{e ≥ i} F[e][j]`. Khi đó:
- `F[i][j] = F[i+1][j] + Suf[i+1][j−1]`
- `Suf[i][j] = F[i][j] + Suf[i+1][j]`

Vì `i` duyệt giảm dần nên khi tính hàng `i` thì hàng `i + 1` đã xong. Ta chỉ cần giữ hai hàng (`F` và `Suf` của `i + 1`) ⇒ bộ nhớ `O(k)`.

In [13]:
def count_suffix(n, k):
    # hàng i = n-1: F[n-1][0] = 1, F[n-1][j>=1] = 0; Suf[n-1] = F[n-1]
    F_next = [1] + [0] * k
    S_next = F_next[:]
    for i in range(n - 2, -1, -1):
        F_cur = [1] + [0] * k
        S_cur = [0] * (k + 1)
        for j in range(1, k + 1):
            F_cur[j] = (F_next[j] + S_next[j - 1]) % MOD
        for j in range(k + 1):
            S_cur[j] = (F_cur[j] + S_next[j]) % MOD
        F_next, S_next = F_cur, S_cur
    return F_next[k]

for (n, k), want in examples:
    print((n, k), count_suffix(n, k), "kỳ vọng", want)

(4, 2) 5 kỳ vọng 5
(3, 1) 3 kỳ vọng 3
(30, 7) 796297179 kỳ vọng 796297179


Code nộp LeetCode của Cách 3:

```python
class Solution:
    def numberOfSets(self, n: int, k: int) -> int:
        MOD = 10**9 + 7
        F_next = [1] + [0] * k
        S_next = F_next[:]
        for i in range(n - 2, -1, -1):
            F_cur = [1] + [0] * k
            for j in range(1, k + 1):
                F_cur[j] = (F_next[j] + S_next[j - 1]) % MOD
            S_next = [(F_cur[j] + S_next[j]) % MOD for j in range(k + 1)]
            F_next = F_cur
        return F_next[k]
```

**Độ phức tạp:** Time `O(n·k)` (≈ `10⁶` bước khi `n = k = 1000`), Space `O(k)`.

### 7.3 Cách 4 — DP theo khe với hai trạng thái "trống / được phủ", `O(n·k)`

Duyệt lần lượt `n − 1` khe. Trước khe đầu tiên: `a[0] = 1` (coi như "trống", chưa mở đoạn nào), mọi giá trị khác bằng 0.

Mỗi khe:
- `na[j] = a[j] + b[j]` — khe mới trống.
- `nb[j] = b[j] + (a[j−1] + b[j−1])` — nối tiếp đoạn cũ (chỉ từ `b[j]`) hoặc mở đoạn mới (từ `a[j−1]`, `b[j−1]`).

Đáp án: `a[k] + b[k]` sau khe cuối. Đây chính là hai mảng `f`, `g` quen thuộc trong các lời giải DP của bài, nhưng đọc theo nghĩa "khe" thì dễ hình dung hơn.

In [14]:
def count_gaps(n, k):
    a = [1] + [0] * k          # khe trước đó trống
    b = [0] * (k + 1)          # khe trước đó được phủ
    for _ in range(n - 1):
        na = [(a[j] + b[j]) % MOD for j in range(k + 1)]
        nb = [0] * (k + 1)
        for j in range(k + 1):
            nb[j] = b[j]
            if j:
                nb[j] = (nb[j] + a[j - 1] + b[j - 1]) % MOD
        a, b = na, nb
    return (a[k] + b[k]) % MOD

for (n, k), want in examples:
    print((n, k), count_gaps(n, k), "kỳ vọng", want)

(4, 2) 5 kỳ vọng 5
(3, 1) 3 kỳ vọng 3
(30, 7) 796297179 kỳ vọng 796297179


Code nộp LeetCode của Cách 4:

```python
class Solution:
    def numberOfSets(self, n: int, k: int) -> int:
        MOD = 10**9 + 7
        a = [1] + [0] * k
        b = [0] * (k + 1)
        for _ in range(n - 1):
            na = [(a[j] + b[j]) % MOD for j in range(k + 1)]
            nb = [b[0]] + [(b[j] + a[j - 1] + b[j - 1]) % MOD for j in range(1, k + 1)]
            a, b = na, nb
        return (a[k] + b[k]) % MOD
```

**Độ phức tạp:** Time `O(n·k)`, Space `O(k)`.

### 7.4 Cách 5 — Tổ hợp `C(n + k − 1, 2k)`, `O(n + k)`

**Song ánh chi tiết.** Liệt kê đầu mút theo thứ tự: `x₁ < x₂ ≤ x₃ < x₄ ≤ … ≤ x₂ₖ₋₁ < x₂ₖ`, với `0 ≤ xᵢ ≤ n − 1`. Đặt `yᵢ = xᵢ + ⌊(i − 1)/2⌋` (đoạn thứ `t` được cộng `t − 1`):
- Trong một đoạn: `y₂ₜ₋₁ < y₂ₜ` vì cộng cùng một lượng.
- Giữa hai đoạn: `x₂ₜ ≤ x₂ₜ₊₁` ⇒ `y₂ₜ = x₂ₜ + t − 1 < x₂ₜ₊₁ + t = y₂ₜ₊₁`.
- `0 ≤ y₁` và `y₂ₖ ≤ (n − 1) + (k − 1) = n + k − 2`.

Ngược lại, mọi dãy tăng nghiêm ngặt `2k` số trong `{0, …, n + k − 2}` trừ ngược ra một tập đoạn hợp lệ. Vậy đáp án = số cách chọn `2k` số từ `n + k − 1` số = `C(n + k − 1, 2k)`.

Tính modulo: `C(N, R) = N! / (R!·(N−R)!)` — dùng giai thừa modulo và nghịch đảo Fermat.

In [15]:
def count_comb_exact(n, k):
    return comb(n + k - 1, 2 * k) % MOD           # Python big int: chính xác rồi mới mod

def count_comb_mod(n, k):
    N, R = n + k - 1, 2 * k
    if R > N:
        return 0
    fact = [1] * (N + 1)
    for i in range(1, N + 1):
        fact[i] = fact[i - 1] * i % MOD
    return fact[N] * pow(fact[R], MOD - 2, MOD) % MOD * pow(fact[N - R], MOD - 2, MOD) % MOD

for (n, k), want in examples:
    print((n, k), count_comb_exact(n, k), count_comb_mod(n, k), "kỳ vọng", want)
print("số thật của ví dụ 3:", comb(36, 14))

(4, 2) 5 5 kỳ vọng 5
(3, 1) 3 3 kỳ vọng 3
(30, 7) 796297179 796297179 kỳ vọng 796297179
số thật của ví dụ 3: 3796297200


In [16]:
# Xem song ánh tận mắt với n = 4, k = 2
def to_y(segs):
    ys = []
    for t, (l, r) in enumerate(segs):
        ys += [l + t, r + t]
    return ys

for segs in brute_sets(4, 2):
    print(segs, "-> y =", to_y(segs))
print("các bộ y là 5 tập con 4 phần tử khác nhau của {0..4}:", sorted(map(tuple, map(to_y, brute_sets(4, 2)))))

[(0, 1), (1, 2)] -> y = [0, 1, 2, 3]
[(0, 1), (1, 3)] -> y = [0, 1, 2, 4]
[(0, 1), (2, 3)] -> y = [0, 1, 3, 4]
[(0, 2), (2, 3)] -> y = [0, 2, 3, 4]
[(1, 2), (2, 3)] -> y = [1, 2, 3, 4]
các bộ y là 5 tập con 4 phần tử khác nhau của {0..4}: [(0, 1, 2, 3), (0, 1, 2, 4), (0, 1, 3, 4), (0, 2, 3, 4), (1, 2, 3, 4)]


Code nộp LeetCode của Cách 5:

```python
class Solution:
    def numberOfSets(self, n: int, k: int) -> int:
        MOD = 10**9 + 7
        N, R = n + k - 1, 2 * k
        fact = [1] * (N + 1)
        for i in range(1, N + 1):
            fact[i] = fact[i - 1] * i % MOD
        return fact[N] * pow(fact[R] * fact[N - R] % MOD, MOD - 2, MOD) % MOD
```

(Trong Python cũng có thể viết gọn `return comb(n + k - 1, 2 * k) % (10**9 + 7)` — số nguyên lớn vẫn nhanh với `N ≤ 1998`.)

**Độ phức tạp:** Time `O(n + k)` cho giai thừa + `O(log MOD)` cho lũy thừa, Space `O(n + k)`.

## 8. TRACE từng bước

Trace **Cách 4** (DP theo khe) với `n = 4, k = 2` (3 khe):

| sau khe | a[0] | a[1] | a[2] | b[0] | b[1] | b[2] | ý nghĩa |
|---|---|---|---|---|---|---|---|
| (khởi tạo) | 1 | 0 | 0 | 0 | 0 | 0 | chưa có khe |
| 1 | 1 | 0 | 0 | 0 | 1 | 0 | `.` hoặc `S` |
| 2 | 1 | 1 | 0 | 0 | 2 | 1 | `..`, `S.` / `.S`, `SC` / `SS` |
| 3 | 1 | 3 | 1 | 0 | 3 | 4 | đáp án `a[2] + b[2] = 1 + 4 = 5` |

In [17]:
def trace_gaps(n, k):
    a = [1] + [0] * k
    b = [0] * (k + 1)
    print(f"khởi tạo   a={a} b={b}")
    for g in range(1, n):
        na = [(a[j] + b[j]) % MOD for j in range(k + 1)]
        nb = [b[0]] + [(b[j] + a[j - 1] + b[j - 1]) % MOD for j in range(1, k + 1)]
        a, b = na, nb
        print(f"sau khe {g} (nối điểm {g-1}-{g}): a={a} b={b}")
    ans = (a[k] + b[k]) % MOD
    print("đáp án = a[k] + b[k] =", ans)
    return ans

assert trace_gaps(4, 2) == 5

khởi tạo   a=[1, 0, 0] b=[0, 0, 0]
sau khe 1 (nối điểm 0-1): a=[1, 0, 0] b=[0, 1, 0]
sau khe 2 (nối điểm 1-2): a=[1, 1, 0] b=[0, 2, 1]
sau khe 3 (nối điểm 2-3): a=[1, 3, 1] b=[0, 3, 4]
đáp án = a[k] + b[k] = 5


In [18]:
def trace_suffix(n, k):
    F_next = [1] + [0] * k
    S_next = F_next[:]
    print(f"i={n-1}: F={F_next} Suf={S_next}")
    for i in range(n - 2, -1, -1):
        F_cur = [1] + [0] * k
        for j in range(1, k + 1):
            F_cur[j] = (F_next[j] + S_next[j - 1]) % MOD
        S_next = [(F_cur[j] + S_next[j]) % MOD for j in range(k + 1)]
        F_next = F_cur
        print(f"i={i}: F={F_next} Suf={S_next}   (F[i][j] = F[i+1][j] + Suf[i+1][j-1])")
    return F_next[k]

assert trace_suffix(4, 2) == 5

i=3: F=[1, 0, 0] Suf=[1, 0, 0]
i=2: F=[1, 1, 0] Suf=[2, 1, 0]   (F[i][j] = F[i+1][j] + Suf[i+1][j-1])
i=1: F=[1, 3, 1] Suf=[3, 4, 1]   (F[i][j] = F[i+1][j] + Suf[i+1][j-1])
i=0: F=[1, 6, 5] Suf=[4, 10, 6]   (F[i][j] = F[i+1][j] + Suf[i+1][j-1])


Đọc trace Cách 3: ở `i = 0`, `F[0][2] = F[1][2] + Suf[1][1] = 1 + (F[1][1] + F[2][1] + F[3][1]) = 1 + (3 + 1 + 0) = 5`. Nghĩa: hoặc không đặt đoạn nào bắt đầu ở 0 (còn 1 cách: `{(1,2),(2,3)}`), hoặc đoạn đầu là `(0,e)` rồi đặt 1 đoạn trên `e..3`.

## 9. Code nộp LeetCode

Chọn Cách 4 (DP theo khe, dễ giải thích, không cần biết tổ hợp) — ổn định `O(n·k)`.

In [19]:
class Solution:
    def numberOfSets(self, n: int, k: int) -> int:
        MOD = 10**9 + 7
        a = [1] + [0] * k          # a[j]: đã mở j đoạn, khe vừa xét trống
        b = [0] * (k + 1)          # b[j]: đã mở j đoạn, khe vừa xét được phủ
        for _ in range(n - 1):
            na = [(a[j] + b[j]) % MOD for j in range(k + 1)]
            nb = [b[0]] + [(b[j] + a[j - 1] + b[j - 1]) % MOD for j in range(1, k + 1)]
            a, b = na, nb
        return (a[k] + b[k]) % MOD

for (n, k), want in examples:
    got = Solution().numberOfSets(n, k)
    print((n, k), got, "OK" if got == want else "SAI")

(4, 2) 5 OK
(3, 1) 3 OK
(30, 7) 796297179 OK


Bản ngắn nhất (Cách 5) cũng hoàn toàn hợp lệ:

```python
from math import comb

class Solution:
    def numberOfSets(self, n: int, k: int) -> int:
        return comb(n + k - 1, 2 * k) % (10**9 + 7)
```

## 10. Vì sao thuật toán đúng

**Cách 4 (khe).**
1. *Song ánh tập đoạn ↔ chuỗi khe.* Mỗi tập đoạn hợp lệ cho đúng một chuỗi trên `{., S, C}`: khe đầu của mỗi đoạn là `S`, các khe còn lại của đoạn là `C`, khe không thuộc đoạn nào là `.`. Vì các đoạn không chồng nhau nên mỗi khe thuộc nhiều nhất một đoạn. Ngược lại, chuỗi hợp lệ (không có `C` đứng đầu hay sau `.`) tách được duy nhất thành các đoạn: mỗi `S` cùng các `C` liền sau. Số đoạn = số `S`.
2. *Invariant.* Sau khi xử lý `g` khe, `a[j]` (`b[j]`) = số tiền tố hợp lệ độ dài `g`, có `j` chữ `S`, ký tự cuối là `.` (là `S`/`C`). Quy nạp theo `g`: ba lựa chọn cho khe mới tương ứng đúng ba công thức chuyển, và ràng buộc "`C` cần khe trước được phủ" thể hiện ở việc `C` chỉ đi từ `b`.
3. Kết thúc: mọi chuỗi độ dài `n − 1` có `k` chữ `S` nằm trong `a[k]` hoặc `b[k]` ⇒ đáp án `a[k] + b[k]`.

**Cách 3.** `F(i, j)` phân loại theo "có đoạn bắt đầu tại `i` không", hai nhóm rời nhau và phủ mọi khả năng ⇒ công thức cộng đúng; tổng hậu tố chỉ là cách tính nhanh cùng một tổng.

**Cách 5.** Phép biến đổi `yᵢ = xᵢ + ⌊(i−1)/2⌋` là song ánh (có phép ngược `xᵢ = yᵢ − ⌊(i−1)/2⌋`, giữ đúng các bất đẳng thức như đã kiểm ở mục 7.4) ⇒ hai tập có cùng lực lượng.

**Modulo.** Chỉ dùng cộng và nhân (Cách 3, 4) nên lấy modulo từng bước không đổi kết quả. Cách 5 dùng chia ⇒ phải nhân với nghịch đảo modulo, hợp lệ vì `10⁹ + 7` nguyên tố và các giai thừa `< p` không chia hết cho `p`.

## 11. Phân tích Time / Space Complexity

Gọi `n` = số điểm (≤ 1000), `k` = số đoạn (≤ n − 1).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Vét cạn | `O(đáp án · n²)` | `O(k)` (đệ quy) | chỉ dùng cho `n ≤ 12` |
| 2. DP thử mọi điểm kết thúc | `O(n²·k)` | `O(n·k)` | `≈ 5·10⁸` khi `n = k = 1000` ⇒ TLE |
| 3. DP + tổng hậu tố | `O(n·k)` | `O(k)` | `≈ 10⁶` phép tính |
| 4. DP theo khe | `O(n·k)` | `O(k)` | như Cách 3, công thức gọn hơn |
| 5. Tổ hợp | `O(n + k)` | `O(n + k)` | nhanh nhất; cần thấy song ánh |

Không có khác biệt average/worst: mọi cách (trừ vét cạn) chạy đúng số bước như nhau cho mọi input cùng `(n, k)`.

In [20]:
for f in (count_suffix, count_gaps, count_comb_mod):
    t = time.perf_counter()
    v = f(1000, 500)
    print(f"{f.__name__:15s} n=1000,k=500 -> {v} trong {time.perf_counter() - t:.3f}s")
t = time.perf_counter(); count_n2k(150, 75)
print(f"count_n2k n=150,k=75 trong {time.perf_counter() - t:.3f}s (n=1000 sẽ chậm ~ (1000/150)^3 ≈ 300 lần)")

count_suffix    n=1000,k=500 -> 70047606 trong 0.051s


count_gaps      n=1000,k=500 -> 70047606 trong 0.068s
count_comb_mod  n=1000,k=500 -> 70047606 trong 0.000s


count_n2k n=150,k=75 trong 0.026s (n=1000 sẽ chậm ~ (1000/150)^3 ≈ 300 lần)


## 12. Bộ test

- Ví dụ đề.
- Edge: `n = 2, k = 1` (đúng 1 cách), `k = n − 1` (mọi khe là một đoạn riêng ⇒ 1 cách), `k = 1` (`C(n, 2)` cách — chọn 2 điểm bất kỳ).
- Đối chiếu tất cả các cách với vét cạn trên mọi `(n, k)` nhỏ và với nhau trên `(n, k)` ngẫu nhiên lớn (seed cố định), đúng ràng buộc `2 ≤ n ≤ 1000`, `1 ≤ k ≤ n − 1`.

In [21]:
FAST = [count_suffix, count_gaps, count_comb_exact, count_comb_mod,
        lambda n, k: Solution().numberOfSets(n, k)]

for (n, k), want in examples:
    for f in FAST + [count_memo, count_n2k]:
        assert f(n, k) == want, (f, n, k)

# edge cases
for f in FAST:
    assert f(2, 1) == 1
    for n in (2, 5, 17, 1000):
        assert f(n, n - 1) == 1            # mỗi khe là một đoạn
        assert f(n, 1) == n * (n - 1) // 2 % MOD
print("ví dụ đề + edge case: OK")

ví dụ đề + edge case: OK


In [22]:
# Mọi (n, k) nhỏ: so với vét cạn
cnt = 0
for n in range(2, 10):
    for k in range(1, n):
        want = brute_count(n, k)
        assert len(gap_strings(n, k)) == want
        for f in FAST + [count_memo, count_n2k]:
            assert f(n, k) == want, (f, n, k)
        cnt += 1
print("đã so", cnt, "cặp (n, k) với vét cạn: OK")

đã so 36 cặp (n, k) với vét cạn: OK


In [23]:
# Ngẫu nhiên lớn, seed cố định: các cách nhanh khớp nhau
rnd = random.Random(1621)
for _ in range(25):
    n = rnd.randint(2, 1000)
    k = rnd.randint(1, n - 1)
    vals = {f(n, k) for f in FAST}
    assert len(vals) == 1, (n, k, vals)
for _ in range(40):
    n = rnd.randint(2, 60); k = rnd.randint(1, n - 1)
    assert count_n2k(n, k) == count_comb_exact(n, k)
print("random tests: OK")

random tests: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Không cho phép chung đầu mút

Đặt đoạn sau bắt đầu từ `r + 1` thay vì `r`.

In [24]:
def wrong_no_share(n, k):
    cnt = 0
    def go(start, left):
        nonlocal cnt
        if left == 0:
            cnt += 1; return
        for l in range(start, n):
            for r in range(l + 1, n):
                go(r + 1, left - 1)          # SAI: cấm chung đầu mút
    go(0, k)
    return cnt

print("sai:", wrong_no_share(4, 2), "| đúng:", brute_count(4, 2))
print("bản sai thực chất đếm C(n, 2k):", comb(4, 4))

sai: 1 | đúng: 5
bản sai thực chất đếm C(n, 2k): 1


**Vì sao sai:** đề nói rõ các đoạn **được phép** chung đầu mút; bản sai chỉ đếm `{(0,1),(2,3)}` và bỏ 4 cách còn lại. **Sửa:** gọi đệ quy với `start = r` (hoặc trong DP: điều kiện `r₁ ≤ l₂`).

### Lỗi 2 — Lấy modulo rồi mới chia (phép chia nguyên trong modulo)

In [25]:
def wrong_div_mod(n, k):
    N, R = n + k - 1, 2 * k
    fact = [1] * (N + 1)
    for i in range(1, N + 1):
        fact[i] = fact[i - 1] * i % MOD
    return fact[N] // (fact[R] * fact[N - R] % MOD) % MOD     # SAI: // sau khi đã mod

print("sai:", wrong_div_mod(30, 7), "| đúng:", count_comb_mod(30, 7))

sai: 0 | đúng: 796297179


**Vì sao sai:** sau khi lấy modulo, `fact[N]` không còn chia hết cho `fact[R]·fact[N−R]` nữa; phép `//` cho kết quả vô nghĩa. **Sửa:** nhân với nghịch đảo `pow(x, MOD - 2, MOD)`, hoặc dùng `math.comb` (số nguyên lớn, chính xác) rồi mới `% MOD`.

### Lỗi 3 — Đệ quy có nhớ quá sâu

In [26]:
def memo_recursive(n, k):
    @lru_cache(maxsize=None)
    def F(i, j):
        if j == 0: return 1
        if i >= n - 1: return 0
        return (F(i + 1, j) + sum(F(e, j - 1) for e in range(i + 1, n))) % MOD
    return F(0, k)

old = sys.getrecursionlimit()
sys.setrecursionlimit(300)          # mô phỏng giới hạn chật (n=1000 sẽ vượt giới hạn mặc định ~1000 như vậy)
try:
    memo_recursive(400, 1)
    print("chạy được")
except RecursionError:
    print("RecursionError: chuỗi F(0)->F(1)->...->F(n-1) sâu ~n tầng, vượt giới hạn")
finally:
    sys.setrecursionlimit(old)
print("bản lặp không bị:", count_suffix(400, 1) == comb(400, 2) % MOD)

RecursionError: chuỗi F(0)->F(1)->...->F(n-1) sâu ~n tầng, vượt giới hạn
bản lặp không bị: True


**Vì sao sai:** `F(i, j)` gọi `F(i + 1, j)` nên độ sâu đệ quy ≈ `n` (cộng thêm khung của `lru_cache`, generator…). Với `n = 1000` dễ vượt giới hạn mặc định. **Sửa:** viết DP lặp (Cách 3, 4) hoặc tăng `sys.setrecursionlimit` (vẫn kém an toàn).

### Lỗi 4 — Lặp `n` khe thay vì `n − 1` khe

In [27]:
def wrong_gap_count(n, k):
    a = [1] + [0] * k; b = [0] * (k + 1)
    for _ in range(n):                     # SAI: n điểm chỉ có n-1 khe
        na = [(a[j] + b[j]) % MOD for j in range(k + 1)]
        nb = [b[0]] + [(b[j] + a[j - 1] + b[j - 1]) % MOD for j in range(1, k + 1)]
        a, b = na, nb
    return (a[k] + b[k]) % MOD

print("sai:", wrong_gap_count(4, 2), "| đúng:", count_gaps(4, 2))

sai: 15 | đúng: 5


**Vì sao sai:** đếm như thể có `n + 1` điểm (kết quả là đáp án của `n + 1`). **Sửa:** số khe = số điểm − 1; luôn tự kiểm `n = 2, k = 1` phải ra `1`.

### Lỗi 5 — Quên điều kiện "phủ ít nhất hai điểm" (cho đoạn độ dài 0)

In [28]:
def wrong_zero_len(n, k):
    cnt = 0
    def go(start, left):
        nonlocal cnt
        if left == 0:
            cnt += 1; return
        for l in range(start, n):
            for r in range(l, n):            # SAI: cho phép r == l
                go(r + (r == l), left - 1)   # (tránh đếm vô hạn đoạn điểm)
    go(0, k)
    return cnt

print("sai:", wrong_zero_len(3, 1), "| đúng:", brute_count(3, 1))

sai: 6 | đúng: 3


**Vì sao sai:** đoạn `(1,1)` chỉ là một điểm, đề yêu cầu phủ ≥ 2 điểm. **Sửa:** `r` chạy từ `l + 1`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đếm số cách … modulo `10⁹ + 7`" + `n ≤ 1000` ⇒ nghĩ ngay DP `O(n²)` hoặc `O(n·k)`.
- Đối tượng là **đoạn / khoảng rời nhau trên đường thẳng** ⇒ quét trái → phải, trạng thái "đang ở trong đoạn hay không".
- Thấy `Σ_{e = i+1}^{n−1} F(e, …)` ⇒ **prefix/suffix sum** để bỏ một vòng lặp.
- Viết vét cạn, in đáp án cho `n, k` nhỏ thành bảng, rồi tra/đoán công thức: bảng của bài này trùng với `C(n + k − 1, 2k)` — mẹo "đoán rồi chứng minh" rất hiệu quả với bài đếm.
- Dãy chỉ số dạng `x₁ < x₂ ≤ x₃ < …` ⇒ kỹ thuật dịch chỉ số (stars and bars) để thành tổ hợp.

In [29]:
# Bảng đáp án nhỏ: hàng n, cột k — thử nhận ra công thức
print("n\\k " + " ".join(f"{k:>5}" for k in range(1, 7)))
for n in range(2, 9):
    row = [brute_count(n, k) if k < n else "" for k in range(1, 7)]
    print(f"{n:>3} " + " ".join(f"{str(v):>5}" for v in row))
print("so với C(n+k-1, 2k):", [comb(n + 2 - 1, 4) for n in range(2, 9)], "(cột k=2)")

n\k     1     2     3     4     5     6
  2     1                              
  3     3     1                        
  4     6     5     1                  
  5    10    15     7     1            
  6    15    35    28     9     1      
  7    21    70    84    45    11     1
  8    28   126   210   165    66    13
so với C(n+k-1, 2k): [0, 1, 5, 15, 35, 70, 126] (cột k=2)


## 15. Biến thể của bài

1. **Không được chung đầu mút** ⇒ chọn `2k` đầu mút phân biệt tăng nghiêm ngặt ⇒ `C(n, 2k)`.
2. **Mỗi đoạn dài ít nhất `L` đơn vị** ⇒ DP theo khe với trạng thái "đã phủ bao nhiêu khe trong đoạn hiện tại" (hoặc dịch chỉ số: `C(n − 1 − k(L−1) + k, 2k)` khi cho chung đầu mút).
3. **Không giới hạn số đoạn** (đếm mọi `k`) ⇒ cộng dồn; DP khe chỉ còn 2 trạng thái.
4. **LC 1641 Count Sorted Vowel Strings** — cũng là stars and bars `C(n + 4, 4)`.
5. **LC 62 Unique Paths**, **LC 1866 Number of Ways to Rearrange Sticks** — đếm tổ hợp/DP modulo cùng họ.
6. **Đoạn trên vòng tròn** — phải xử lý thêm việc đoạn đi qua điểm 0.

In [30]:
# Biến thể 1: kiểm chứng C(n, 2k) cho "không chung đầu mút"
for n in range(2, 9):
    for k in range(1, n):
        assert wrong_no_share(n, k) == comb(n, 2 * k)
print("Biến thể 'không chung đầu mút' = C(n, 2k): OK")

Biến thể 'không chung đầu mút' = C(n, 2k): OK


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | sinh từng tập đoạn trái → phải | rất lớn | `O(k)` | kiểm thử, `n ≤ 12` |
| 2. DP thử điểm kết thúc | `F(i,j) = F(i+1,j) + Σ F(e,j−1)` | `O(n²k)` | `O(nk)` | bước đệm để thấy tổng liên tiếp |
| 3. DP + tổng hậu tố | thay `Σ` bằng `Suf` | `O(nk)` | `O(k)` | khi đã có DP `O(n²k)` và cần tối ưu |
| 4. DP theo khe | 2 trạng thái trống / được phủ | `O(nk)` | `O(k)` | lời giải phỏng vấn tự nhiên nhất |
| 5. Tổ hợp | song ánh → `C(n+k−1, 2k)` | `O(n+k)` | `O(n+k)` | khi nhận ra công thức; code ngắn nhất |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `numberOfSets(5, 1)`, `numberOfSets(5, 4)`, `numberOfSets(2, 1)`, `numberOfSets(6, 3)`.

**Bài 2 (tính tay):** liệt kê đủ các tập với `n = 4, k = 1` và đếm số tập với `n = 5, k = 2` bằng bảng `a/b` của Cách 4.

**Bài 3 (điền TODO):** hoàn thiện hàm `step(a, b)` thực hiện **một** khe của Cách 4.

**Bài 4 (tự viết lại):** viết `ncr_mod(N, R)` tính `C(N, R) mod 10⁹+7` bằng giai thừa + nghịch đảo Fermat, không dùng `math.comb`.

**Bài 5 (mini challenge):** đếm số cách vẽ đúng `k` đoạn không chồng nhau **và không được chung đầu mút**, mỗi đoạn phủ ≥ 2 điểm, bằng DP `O(n·k)` (không dùng công thức), rồi đối chiếu với `C(n, 2k)`.

In [31]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def step(a, b):
    k = len(a) - 1
    na = [0] * (k + 1)
    nb = [0] * (k + 1)
    # TODO: na[j] = khe mới trống
    # TODO: nb[j] = nối tiếp (từ b[j]) + mở mới (từ a[j-1], b[j-1])
    return na, nb

In [32]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def ncr_mod(N, R):
    # TODO
    pass

def count_no_share(n, k):
    # TODO: DP theo khe nhưng giữa hai đoạn phải có ít nhất... (gợi ý ở mục 18)
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: dùng công thức `C(n + k − 1, 2k)`. Mức 2: `k = 1` ⇒ `C(n, 2)`; `k = n − 1` ⇒ 1. Mức 3: `10, 1, 1, C(8, 6) = 28`.

**Bài 2.** Mức 1: `k = 1` chỉ là chọn hai điểm làm đầu mút. Mức 2: `n = 4` có `C(4,2) = 6` đoạn. Mức 3: với `n = 5, k = 2` chạy 4 khe, cuối cùng `a[2] + b[2] = 15`.

**Bài 3.** Mức 1: ba lựa chọn cho khe mới: `.`, `S`, `C`. Mức 2: `.` nhận từ cả `a[j]`, `b[j]`; `C` chỉ từ `b[j]`. Mức 3: `S` từ `a[j−1] + b[j−1]`, nhớ `% MOD`.

**Bài 4.** Mức 1: `fact[i] = fact[i−1]·i % MOD`. Mức 2: nghịch đảo `pow(x, MOD − 2, MOD)`. Mức 3: xử lý `R < 0` hoặc `R > N` ⇒ 0.

**Bài 5.** Mức 1: "không chung đầu mút" nghĩa là sau khi một đoạn kết thúc ở điểm `r`, đoạn mới phải bắt đầu ở `≥ r + 1` ⇒ khe ngay sau đoạn phải **trống**. Mức 2: khi đó `S` chỉ được đi sau `.` (hoặc đứng đầu), không đi sau `S`/`C`. Mức 3: `nb[j] = b[j] + a[j−1]` (bỏ `b[j−1]`).

## 19. Đáp án tham khảo cho bài tập

In [33]:
# Bài 1
b1 = [Solution().numberOfSets(n, k) for n, k in [(5, 1), (5, 4), (2, 1), (6, 3)]]
print("Bài 1:", b1)
assert b1 == [10, 1, 1, 28]

# Bài 2
print("Bài 2 (n=4,k=1):", brute_sets(4, 1))
assert len(brute_sets(4, 1)) == 6
assert trace_gaps(5, 2) == 15

Bài 1: [10, 1, 1, 28]
Bài 2 (n=4,k=1): [[(0, 1)], [(0, 2)], [(0, 3)], [(1, 2)], [(1, 3)], [(2, 3)]]
khởi tạo   a=[1, 0, 0] b=[0, 0, 0]
sau khe 1 (nối điểm 0-1): a=[1, 0, 0] b=[0, 1, 0]
sau khe 2 (nối điểm 1-2): a=[1, 1, 0] b=[0, 2, 1]
sau khe 3 (nối điểm 2-3): a=[1, 3, 1] b=[0, 3, 4]
sau khe 4 (nối điểm 3-4): a=[1, 6, 5] b=[0, 4, 10]
đáp án = a[k] + b[k] = 15


In [34]:
# Bài 3
def step_sol(a, b):
    k = len(a) - 1
    na = [(a[j] + b[j]) % MOD for j in range(k + 1)]
    nb = [b[0]] + [(b[j] + a[j - 1] + b[j - 1]) % MOD for j in range(1, k + 1)]
    return na, nb

def count_by_step(n, k):
    a, b = [1] + [0] * k, [0] * (k + 1)
    for _ in range(n - 1):
        a, b = step_sol(a, b)
    return (a[k] + b[k]) % MOD

for n in range(2, 40):
    for k in range(1, n):
        assert count_by_step(n, k) == count_comb_exact(n, k)
print("Bài 3: OK")

Bài 3: OK


In [35]:
# Bài 4
def ncr_mod_sol(N, R):
    if R < 0 or R > N:
        return 0
    fact = [1] * (N + 1)
    for i in range(1, N + 1):
        fact[i] = fact[i - 1] * i % MOD
    return fact[N] * pow(fact[R], MOD - 2, MOD) % MOD * pow(fact[N - R], MOD - 2, MOD) % MOD

rnd = random.Random(4)
for _ in range(200):
    N = rnd.randint(0, 2000); R = rnd.randint(-2, N + 2)
    assert ncr_mod_sol(N, R) == (comb(N, R) % MOD if 0 <= R <= N else 0)
print("Bài 4: OK")

Bài 4: OK


In [36]:
# Bài 5
def count_no_share_sol(n, k):
    a = [1] + [0] * k
    b = [0] * (k + 1)
    for _ in range(n - 1):
        na = [(a[j] + b[j]) % MOD for j in range(k + 1)]
        nb = [b[0]] + [(b[j] + a[j - 1]) % MOD for j in range(1, k + 1)]   # S chỉ sau '.'
        a, b = na, nb
    return (a[k] + b[k]) % MOD

for n in range(2, 9):
    for k in range(1, n):
        assert count_no_share_sol(n, k) == wrong_no_share(n, k) == comb(n, 2 * k) % MOD
rnd = random.Random(5)
for _ in range(20):
    n = rnd.randint(2, 800); k = rnd.randint(1, n - 1)
    assert count_no_share_sol(n, k) == comb(n, 2 * k) % MOD
print("Bài 5: OK")

Bài 5: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template DP "khe" cho đoạn rời nhau (được chung đầu mút):**

```python
a = [1] + [0] * k          # khe trước trống
b = [0] * (k + 1)          # khe trước được phủ
for _ in range(n - 1):     # n điểm -> n-1 khe
    na = [(a[j] + b[j]) % MOD for j in range(k + 1)]
    nb = [b[0]] + [(b[j] + a[j-1] + b[j-1]) % MOD for j in range(1, k + 1)]
    a, b = na, nb
return (a[k] + b[k]) % MOD
```

**Công thức:** `answer = C(n + k − 1, 2k) mod 10⁹+7` (không chung đầu mút: `C(n, 2k)`).

**Checklist:**
- [ ] Đã cho phép hai đoạn **chung đầu mút** (`r₁ ≤ l₂`)?
- [ ] Mỗi đoạn phủ **≥ 2 điểm** (`l < r`)?
- [ ] Số khe là `n − 1`, không phải `n`?
- [ ] Lấy `% MOD` sau mỗi phép cộng/nhân; **không** chia sau khi đã mod?
- [ ] Không dùng đệ quy sâu `~n` tầng?
- [ ] Đã thử `n = 2, k = 1 → 1`, `k = n − 1 → 1`, `k = 1 → C(n,2)`, ví dụ `n = 30, k = 7 → 796297179`?